In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START
from langchain_ollama import ChatOllama
from rich import print

In [2]:


class State(TypedDict):
    topic: str
    answer: str


llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)


def generate(state: State):
    response = llm.invoke(
        f"Write a short paragraph about {state['topic']}"
    )

    return {"answer": response.content}


graph = (
    StateGraph(State)
    .add_node("generate", generate)
    .add_edge(START, "generate")
    .compile()
)

for chunk in graph.stream(
    {"topic": "Naruto"},
    stream_mode="messages",
    version="v2"
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]

        if msg.content:
            print(msg.content, end="", flush=True)

Naruto is a beloved Japanese manga and anime series created by Mas

ashi Kishimoto, first serialized in Shonen Jump in 1999. The story follows Naruto Uzumaki, a young ninja with 
dreams of becoming

the Hokage—the leader and strongest ninja of his village—while harboring the Nine-Tailed Fox spirit sealed within 
him

. Throughout the series, Naruto forms deep bonds with his teammates Sasuke Uchiha and Sakura Haruno, as well as his
mentor Kakashi Hatake, and faces

countless challenges that test his perseverance, friendships, and beliefs. Known for its themes of determination, 
redemption, and the power of never giving up,

Naruto became a global phenomenon and has inspired numerous sequels, movies, and a new generation of fans. Its 
sequel, *Boruto: Naruto

Next Generations*, continues the legacy by following the adventures of Naruto's son.

In [3]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    topic: str
    refined_topic: str
    answer: str


def refine(state):
    return {
        "refined_topic":
        state["topic"] + " and AI"
    }


def generate(state):
    return {
        "answer":
        f"Generated content about {state['refined_topic']}"
    }


graph = (
    StateGraph(State)
    .add_node("refine", refine)
    .add_node("generate", generate)
    .add_edge(START, "refine")
    .add_edge("refine", "generate")
    .add_edge("generate", END)
    .compile()
)

for chunk in graph.stream(
    {"topic": "Naruto"},
    stream_mode="updates",
    version="v2"
):
    print(chunk)

{'type': 'updates', 'ns': (), 'data': {'refine': {'refined_topic': 'Naruto and AI'}}}

{'type': 'updates', 'ns': (), 'data': {'generate': {'answer': 'Generated content about Naruto and AI'}}}

In [4]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama


# ---------------------------
# State Definition
# ---------------------------
class State(TypedDict):
    topic: str
    refined_topic: str
    answer: str


# ---------------------------
# LLM
# ---------------------------
llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)


# ---------------------------
# Node 1
# ---------------------------
def refine_topic(state: State):

    print("\n[Node] refine_topic executed")

    return {
        "refined_topic": f"{state['topic']} and Artificial Intelligence"
    }


# ---------------------------
# Node 2
# ---------------------------
def generate_answer(state: State):

    print("\n[Node] generate_answer executed")

    response = llm.invoke(
        f"""
        Write a short paragraph about:
        {state['refined_topic']}
        """
    )

    return {
        "answer": response.content
    }


# ---------------------------
# Build Graph
# ---------------------------
graph = (
    StateGraph(State)
    .add_node("refine_topic", refine_topic)
    .add_node("generate_answer", generate_answer)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_answer")
    .add_edge("generate_answer", END)
    .compile()
)


# ---------------------------
# Stream Full State
# ---------------------------
for chunk in graph.stream(
    {
        "topic": "1+1=10 then 1+1+1=?",
        "refined_topic": "",
        "answer": ""
    },
    stream_mode="values",
    version="v2"
):

    if chunk["type"] == "values":

        print("\n" + "=" * 50)
        print("FULL STATE")
        print("=" * 50)

        print(chunk["data"])

==================================================

FULL STATE

==================================================

{'topic': '1+1=10 then 1+1+1=?', 'refined_topic': '', 'answer': ''}

[Node] refine_topic executed

...

[Node] generate_answer executed

==================================================

FULL STATE

==================================================

{
    'topic': '1+1=10 then 1+1+1=?',
    'refined_topic': '1+1=10 then 1+1+1=? and Artificial Intelligence',
    'answer': 'In binary arithmetic, 1+1=10, meaning the sum of two ones is written as "10" in base-two (which 
equals 2 in our everyday decimal system). Following that same logic, 1+1+1 would equal 11 in binary, or 3 in 
decimal. This simple shift in number systems beautifully mirrors the foundation of Artificial Intelligence: just as
AI must translate human concepts into a language machines understand—ones and zeros—it also invites us to rethink 
familiar problems from entirely new perspectives. Behind every chatbot, image generator, and recommendation engine 
lies an unfathomable ocean of binary calculations, where something as basic as 1+1 becomes the first step toward 
machines that can reason, learn, and even create. In a way, AI itself is the product of millions of "10s" stacked 
together, proving that even the simplest rules, when repeated at scale, can give rise to extraordinary 
intelligence.'
}

In [5]:
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

checkpoint=InMemorySaver()
config={
    "configurable": {
        "thread_id": "thread_123"
    }
}

# ---------------------------
# State Definition
# ---------------------------
class State(TypedDict):
    topic: str
    refined_topic: str
    answer: str

# ---------------------------
# LLM
# ---------------------------
llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)


# ---------------------------
# Node 1
# ---------------------------
def refine_topic(state: State):

    print("\n[Node] refine_topic executed")

    return {
        "refined_topic": f"{state['topic']} and Artificial Intelligence"
    }


# ---------------------------
# Node 2
# ---------------------------
def generate_answer(state: State):

    print("\n[Node] generate_answer executed")

    response = llm.invoke(
        f"""
        Write a short paragraph about:
        {state['refined_topic']}
        """
    )

    return {
        "answer": response.content
    }


# ---------------------------
# Build Graph
# ---------------------------
graph = (
    StateGraph(State)
    .add_node("refine_topic", refine_topic)
    .add_node("generate_answer", generate_answer)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_answer")
    .add_edge("generate_answer", END)
    .compile(checkpointer=checkpoint)
)


In [6]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="updates",
    version="v2",
    config={
    "configurable": {
        "thread_id": "thread_123"
    }
}

):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node `{node_name}` updated: {state}")

[Node] refine_topic executed

Node `refine_topic` updated: {'refined_topic': 'ice cream and Artificial Intelligence'}

[Node] generate_answer executed

Node `generate_answer` updated: {'answer': "Ice cream and artificial intelligence might seem like an unlikely pair,
but they share a surprisingly sweet connection. AI is now being used to develop new and unique ice cream flavors by
analyzing vast databases of ingredient combinations, predicting which pairings will delight our taste buds, and 
even optimizing recipes for the perfect texture and creaminess. Some companies use machine learning algorithms to 
study customer preferences, helping them create personalized recommendations or even on-demand, 3D-printed ice 
cream treats. Beyond the freezer, AI-powered robots are assisting in production lines, ensuring consistent quality,
while creative tools like generative AI are helping dessert brands design eye-catching packaging and marketing 
campaigns. So the next time you enjoy a scoop of your favorite flavor, there's a good chance that a touch of 
artificial intelligence helped make that moment possible. 🍦🤖"}

In [7]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="values",
    version="v2",
    config=config
):
    if chunk["type"] == "values":
        print(f"Final state: {chunk['data']}")

Final state: {'topic': 'ice cream', 'refined_topic': 'ice cream and Artificial Intelligence', 'answer': "Ice cream 
and artificial intelligence might seem like an unlikely pair, but they share a surprisingly sweet connection. AI is
now being used to develop new and unique ice cream flavors by analyzing vast databases of ingredient combinations, 
predicting which pairings will delight our taste buds, and even optimizing recipes for the perfect texture and 
creaminess. Some companies use machine learning algorithms to study customer preferences, helping them create 
personalized recommendations or even on-demand, 3D-printed ice cream treats. Beyond the freezer, AI-powered robots 
are assisting in production lines, ensuring consistent quality, while creative tools like generative AI are helping
dessert brands design eye-catching packaging and marketing campaigns. So the next time you enjoy a scoop of your 
favorite flavor, there's a good chance that a touch of artificial intelligence helped make that moment possible. 
🍦🤖"}

[Node] refine_topic executed

Final state: {'topic': 'ice cream', 'refined_topic': 'ice cream and Artificial Intelligence', 'answer': "Ice cream 
and artificial intelligence might seem like an unlikely pair, but they share a surprisingly sweet connection. AI is
now being used to develop new and unique ice cream flavors by analyzing vast databases of ingredient combinations, 
predicting which pairings will delight our taste buds, and even optimizing recipes for the perfect texture and 
creaminess. Some companies use machine learning algorithms to study customer preferences, helping them create 
personalized recommendations or even on-demand, 3D-printed ice cream treats. Beyond the freezer, AI-powered robots 
are assisting in production lines, ensuring consistent quality, while creative tools like generative AI are helping
dessert brands design eye-catching packaging and marketing campaigns. So the next time you enjoy a scoop of your 
favorite flavor, there's a good chance that a touch of artificial intelligence helped make that moment possible. 
🍦🤖"}

[Node] generate_answer executed

Final state: {'topic': 'ice cream', 'refined_topic': 'ice cream and Artificial Intelligence', 'answer': 'Ice cream 
and artificial intelligence might seem like an unlikely pair, but the two are increasingly intersecting in 
delicious ways. AI is now being used by major brands to analyze consumer preferences, predict trending flavors, and
even develop entirely new recipes by studying combinations of ingredients that have never been tried before. Some 
innovative companies have employed machine learning algorithms to optimize production processes, ensuring the 
perfect texture and temperature consistency from batch to batch. Beyond the factory, AI-powered recommendation 
systems help customers discover new flavors based on their past purchases, while chatbots take orders and answer 
questions with surprising charm. On the creative side, AI has even generated imaginative new concepts like "wasabi 
lavender swirl" or "bacon matcha," pushing the boundaries of what ice cream can be. Together, cold creamy treats 
and cutting-edge technology are scooping up a future where innovation never melts away.'}

In [8]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="messages",
    version="v2",config={
    "configurable": {
        "thread_id": "thread_123"
    }
}

):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        if msg.content:
            print(f"Message: {msg.content}")

[Node] refine_topic executed

[Node] generate_answer executed

Message: Ice cream and artificial intelligence may seem like an unlikely pair, but they share a surprising amount 
in common. Both are crafted through careful experimentation

Message:  with flavors, textures, and combinations, and both aim to deliver a delightful, personalized experience. 
Today, AI is revolutionizing the ice cream industry by analyzing

Message:  vast datasets of consumer preferences to predict trending flavors, optimizing production processes to 
achieve the perfect creamy consistency

Message: , and even powering creative tools that help artisans develop unique new taste combinations. From AI

Message: -driven flavor generators that suggest exotic pairings like honey-lavender or matcha-raspberry, to smart 
free

Message: zers that monitor inventory in real time at ice cream shops, technology is making our favorite frozen 
treat more innovative and accessible

Message:  than ever. In a way, the future of ice cream is being scooped out by algorithms that blend science with a

Message:  touch of sweetness.

In [9]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph

# Initialize Checkpointer
checkpoint = InMemorySaver()
config = {
    "configurable": {
        "thread_id": "thread_123"
    }
}

# ---------------------------
# State Definition
# ---------------------------
class State(TypedDict):
    topic: str
    refined_topic: str
    answer: str

# ---------------------------
# LLM
# ---------------------------
llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

# ---------------------------
# Node 1
# ---------------------------
def refine_topic(state: State):
    print("\n[Node Running] refine_topic...")
    return {
        "refined_topic": f"{state['topic']} and Artificial Intelligence"
    }

# ---------------------------
# Node 2
# ---------------------------
def generate_answer(state: State):
    print("\n[Node Running] generate_answer...")
    response = llm.invoke(
        f"""
        Write a short paragraph about:
        {state['refined_topic']}
        """
    )
    return {
        "answer": response.content
    }

# ---------------------------
# Build Graph
# ---------------------------
graph = (
    StateGraph(State)
    .add_node("refine_topic", refine_topic)
    .add_node("generate_answer", generate_answer)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_answer")
    .add_edge("generate_answer", END)
    .compile(checkpointer=checkpoint)
)

for chunk in graph.stream(
    {"topic": "Healthcare"},
    config=config,
    stream_mode="checkpoints",
    version="v2",
):
    if chunk["type"] == "checkpoints":
        # FIX: chunk["data"] is a dict, not an object
        snapshot = chunk["data"]
        
        print("\n⚡ [STREAM] New Checkpoint Emitted")
        # Use dictionary lookups instead of dot notation
        print(f"   Checkpoint ID : {snapshot['config']['configurable'].get('checkpoint_id')}")
        print(f"   State Values  : {snapshot['values']}")
        print(f"   Next Node     : {snapshot.get('next', ())}")
        print("-" * 50)


⚡ [STREAM] New Checkpoint Emitted

Checkpoint ID : 1f15f3c3-70a8-628a-bfff-c6556d6e2114

State Values  : {}

Next Node     : ['__start__']

--------------------------------------------------

...

⚡ [STREAM] New Checkpoint Emitted

Checkpoint ID : 1f15f3c3-bba6-616c-8002-fbbf9f74337c

State Values  : {'topic': 'Healthcare', 'refined_topic': 'Healthcare and Artificial Intelligence', 'answer': 
'Artificial Intelligence is rapidly transforming the healthcare industry, revolutionizing how medical professionals
diagnose, treat, and manage patient care. AI-powered tools can analyze vast amounts of medical data, including 
imaging scans, genetic information, and electronic health records, to identify patterns and detect diseases earlier
and more accurately than traditional methods. Machine learning algorithms assist radiologists in spotting tumors, 
help pathologists analyze tissue samples, and enable predictive models that anticipate patient deterioration before
critical events occur. Beyond diagnostics, AI is streamlining administrative tasks, accelerating drug discovery, 
and personalizing treatment plans based on individual patient profiles. However, the integration of AI in 
healthcare also raises important ethical considerations, including data privacy, algorithmic bias, and the need to 
maintain the human element in patient care. As technology continues to evolve, the collaboration between AI systems
and healthcare professionals holds tremendous promise for improving outcomes, reducing costs, and making quality 
care more accessible worldwide.'}

Next Node     : []

--------------------------------------------------

In [10]:
from typing import TypedDict
from langgraph.graph import START, StateGraph
from langchain_ollama import ChatOllama

# 1. Initialize Ollama instead of OpenAI
# Change "llama3" to whichever model you have downloaded locally (e.g., "mistral", "phi3")
model = ChatOllama(model="minimax-m3:cloud", temperature=0.7)


class State(TypedDict):
    topic: str
    joke: str
    poem: str


def write_joke(state: State):
    topic = state["topic"]
    joke_response = model.invoke(
        [{"role": "user", "content": f"Write a joke about {topic}"}]
    )
    return {"joke": joke_response.content}


def write_poem(state: State):
    topic = state["topic"]
    poem_response = model.invoke(
        [{"role": "user", "content": f"Write a short poem about {topic}"}]
    )
    return {"poem": poem_response.content}


# 2. Build the Parallel Graph
# LangGraph allows passing the function directly to add_edge if named identically
graph = (
    StateGraph(State)
    .add_node(write_joke)
    .add_node(write_poem)
    # Write both the joke and the poem concurrently
    .add_edge(START, write_joke.__name__)
    .add_edge(START, write_poem.__name__)
    .compile()
)

# 3. Stream Messages Token-by-Token
# This remains identical; ChatOllama supports the "messages" streaming mode out of the box
print("Streaming tokens from 'write_poem' node only:\n")
for chunk in graph.stream(
    {"topic": "cats"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the langgraph_node field in the metadata
        # to only include the tokens from the write_poem node
        if msg.content and metadata.get("langgraph_node") == "write_poem":
            print(msg.content, end="|", flush=True)

# print at once
print()

Streaming tokens from 'write_poem' node only:

**|

Whiskered Shadows**

In pools of sun they stretch and sprawl,
Soft as dusk|

, silent as a call.
Eyes like moons, both gold and green,
Watch the world they've|

always seen.

They leave no print on quiet floors,
Yet|

rule the house through closed doors.
A flick of tail, a velvet yawn—
The kings of every|

break of dawn.|

In [11]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph

# Initialize Checkpointer
checkpoint = InMemorySaver()
config = {"configurable": {"thread_id": "thread_123"}}

# State Definition
class State(TypedDict):
    topic: str
    refined_topic: str
    answer: str

# LLM 
llm = ChatOllama(model="minimax-m3:cloud", temperature=0)

# Nodes
def refine_topic(state: State):
    return {"refined_topic": f"{state['topic']} and Artificial Intelligence"}

def generate_answer(state: State):
    response = llm.invoke(f"Write a short paragraph about: {state['refined_topic']}")
    return {"answer": response.content}

# Compile Graph
graph = (
    StateGraph(State)
    .add_node("refine_topic", refine_topic)
    .add_node("generate_answer", generate_answer)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_answer")
    .add_edge("generate_answer", END)
    .compile(checkpointer=checkpoint)
)

# ------------------------------------------------------------------
# MULTIPLE STREAMING IMPLEMENTATION
# ------------------------------------------------------------------
print("Starting Multiple Streaming Mode...\n")

for chunk in graph.stream(
    {"topic": "Healthcare"},
    config=config,
    stream_mode=["messages", "checkpoints"],  # <--- Pass a list of modes here
    version="v2",
):
    # 1. Handle Checkpoint Streams
    if chunk["type"] == "checkpoints":
        snapshot = chunk["data"]
        print(f"\n\n💾 [CHECKPOINT SAVED] Next Node: {snapshot.get('next') or 'Graph Finished!'}")
        print(f"   Current Keys in DB State: {list(snapshot['values'].keys())}")
        print("-" * 60)

    # 2. Handle Token/Message Streams
    elif chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Only print tokens originating from the 'generate_answer' LLM node
        if msg.content and metadata.get("langgraph_node") == "generate_answer":
            print(msg.content, end="", flush=True)


Starting Multiple Streaming Mode...

💾 [CHECKPOINT SAVED] Next Node: ['__start__']

Current Keys in DB State: []

------------------------------------------------------------

💾 [CHECKPOINT SAVED] Next Node: ['refine_topic']

...

patient outcomes are also transforming the industry. However, the integration of AI in healthcare raises important
considerations regarding data privacy, ethical

use, and the need for human oversight to ensure that technology complements rather than replaces the 
compassionate, judgment-based care that defines medicine.

💾 [CHECKPOINT SAVED] Next Node: Graph Finished!

Current Keys in DB State: ['topic', 'refined_topic', 'answer']

------------------------------------------------------------

# Multi Node Streaming

## sequential node

In [12]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import START, END, StateGraph
from rich.console import Console

# Setup Rich console for distinct node colors
console = Console()

class State(TypedDict):
    topic: str
    joke: str
    poem: str

# Use an LLM that supports fast token streaming
llm = ChatOllama(model="minimax-m3:cloud", temperature=0.7)

# --- Node 1 ---
def generate_joke(state: State):
    response = llm.invoke(f"Tell a funny short joke about {state['topic']}")
    return {"joke": response.content}

# --- Node 2 ---
def generate_poem(state: State):
    response = llm.invoke(f"Write a short 4-line poem about {state['topic']}")
    return {"poem": response.content}

# --- Construct Parallel Graph ---
builder = StateGraph(State)
builder.add_node("joke_node", generate_joke)
builder.add_node("poem_node", generate_poem)

# Parallel branch out from START
builder.add_edge(START, "joke_node")
builder.add_edge("joke_node", "poem_node")

builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

graph = builder.compile()

# ------------------------------------------------------------------
# MULTI-NODE STREAMING CONSUMER
# ------------------------------------------------------------------
print("Executing sequential nodes and streaming multi-node tokens:\n")

for chunk in graph.stream(
    {"topic": "robots"},
    stream_mode="messages",
    version="v2"
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        node_name = metadata.get("langgraph_node")  # Identifies which node generated this token
        
        if msg.content:
            # Route and color-code tokens based on their originating node
            if node_name == "joke_node":
                console.print(msg.content, style="bold yellow", end="")
            elif node_name == "poem_node":
                console.print(msg.content, style="bold cyan", end="")


Executing sequential nodes and streaming multi-node tokens:

Why did

 the robot go broke?

Because it spent all its cache! 🤖💰

Want another one? I've got a

 whole *assembly line* of them. 😄

In circuits deep and steel so bright,
Robots dream in code by

 night.
Minds of metal, hearts of light,
They dance through days in pure delight.

## parallel Node

In [13]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import START, END, StateGraph
from rich.console import Console

# Setup Rich console for distinct node colors
console = Console()

class State(TypedDict):
    topic: str
    joke: str
    poem: str

# Use an LLM that supports fast token streaming
llm = ChatOllama(model="minimax-m3:cloud", temperature=0.7)

# --- Node 1 ---
def generate_joke(state: State):
    response = llm.invoke(f"Tell a funny short joke about {state['topic']}")
    return {"joke": response.content}

# --- Node 2 ---
def generate_poem(state: State):
    response = llm.invoke(f"Write a short 4-line poem about {state['topic']}")
    return {"poem": response.content}

# --- Construct Parallel Graph ---
builder = StateGraph(State)
builder.add_node("joke_node", generate_joke)
builder.add_node("poem_node", generate_poem)

# Parallel branch out from START
builder.add_edge(START, "joke_node")
builder.add_edge(START, "poem_node")

builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

graph = builder.compile()

# ------------------------------------------------------------------
# MULTI-NODE STREAMING CONSUMER
# ------------------------------------------------------------------
print("Executing parallel nodes and streaming multi-node tokens:\n")

for chunk in graph.stream(
    {"topic": "robots"},
    stream_mode="messages",
    version="v2"
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        node_name = metadata.get("langgraph_node")  # Identifies which node generated this token
        
        if msg.content:
            # Route and color-code tokens based on their originating node
            if node_name == "joke_node":
                console.print(msg.content, style="bold yellow", end="")
            elif node_name == "poem_node":
                console.print(msg.content, style="bold cyan", end="")

Executing parallel nodes and streaming multi-node tokens:

Why did the robot go on a diet?

Because it had too many *bytes*!

**Steel and Soul**

In circuits deep

 🤖🍕

 and glowing eyes,
They dream of stars and silent skies.
With servos soft, they hum and glide

—
Our metal friends, both heart and guide.

In [14]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver  # Added for checkpoints mode to work
from rich.console import Console

# Setup Rich console for distinct styling
console = Console()

class State(TypedDict):
    topic: str
    joke: str
    poem: str

# Use an LLM that supports fast token streaming
llm = ChatOllama(model="minimax-m3:cloud", temperature=0.7)

# --- Node 1 ---
def generate_joke(state: State):
    response = llm.invoke(f"Tell a funny short joke about {state['topic']}")
    return {"joke": response.content}

# --- Node 2 ---
def generate_poem(state: State):
    response = llm.invoke(f"Write a short 4-line poem about {state['topic']}")
    return {"poem": response.content}

# --- Construct Parallel Graph ---
builder = StateGraph(State)
builder.add_node("joke_node", generate_joke)
builder.add_node("poem_node", generate_poem)

builder.add_edge(START, "joke_node")
builder.add_edge(START, "poem_node")
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

# Crucial: Must compile with a checkpointer for `stream_mode="checkpoints"` to function
memory_saver = InMemorySaver()
graph = builder.compile(checkpointer=memory_saver)

config = {"configurable": {"thread_id": "multi_stream_demo"}}

# ------------------------------------------------------------------
# ALL-IN-ONE STREAMING CONSUMER
# ------------------------------------------------------------------
console.print("[bold green]Starting Graph with ALL Stream Modes (messages, updates, values, checkpoints)...[/bold green]\n")

for chunk in graph.stream(
    {"topic": "robots"},
    config=config,
    stream_mode=["messages", "updates", "values", "checkpoints"], # Passing all 4 modes
    version="v2"
):
    # ------------------------------------------------------------------
    # 1. MESSAGES MODE (Token-by-token streaming)
    # ------------------------------------------------------------------
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        node_name = metadata.get("langgraph_node")
        
        if msg.content:
            if node_name == "joke_node":
                console.print(msg.content, style="bold yellow", end="")
            elif node_name == "poem_node":
                console.print(msg.content, style="bold cyan", end="")

    # ------------------------------------------------------------------
    # 2. UPDATES MODE (Triggered when a specific node completes)
    # ------------------------------------------------------------------
    elif chunk["type"] == "updates":
        node_updates = chunk["data"]
        for node_name, updates in node_updates.items():
            console.print(f"\n\n[bold orange3]✨ [UPDATES] Node `{node_name}` finished executing![/bold orange3]")
            console.print(f"   Delta keys changed: {list(updates.keys())}")

    # ------------------------------------------------------------------
    # 3. VALUES MODE (Triggered when state updates, showing full state snapshot)
    # ------------------------------------------------------------------
    elif chunk["type"] == "values":
        current_state_values = chunk["data"]
        console.print(f"\n[bold magenta]📊 [VALUES] Current combined State dictionary snapshot:[/bold magenta]")
        console.print(f"   State keys present: {list(current_state_values.keys())}")

    # ------------------------------------------------------------------
    # 4. CHECKPOINTS MODE (Triggered when saved to the database)
    # ------------------------------------------------------------------
    elif chunk["type"] == "checkpoints":
        snapshot_dict = chunk["data"]
        console.print(f"\n[bold blue]💾 [CHECKPOINTS] Database entry saved![/bold blue]")
        console.print(f"   ID: {snapshot_dict['config']['configurable'].get('checkpoint_id')}")
        console.print(f"   Next node up for processing: {snapshot_dict.get('next', ())}")
        console.print("-" * 60)


Starting Graph with ALL Stream Modes (messages, updates, values, checkpoints)...

💾 [CHECKPOINTS] Database entry saved!

ID: 1f15f3c4-ee7e-6539-bfff-74969ad21872

Next node up for processing: ['__start__']

------------------------------------------------------------

...

State keys present: ['topic', 'joke', 'poem']

💾 [CHECKPOINTS] Database entry saved!

ID: 1f15f3c5-1b73-64d7-8001-7a0b394b0377

Next node up for processing: []

------------------------------------------------------------

# Multi mode streaming

In [15]:
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

# LLM
llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0.7
)

# State
class State(TypedDict):
    topic: str
    poem: str
    joke: str

# Node 1: Generate Poem
def generate_poem(state: State):
    response = llm.invoke(
        f"Write a short poem about {state['topic']}."
    )

    return {
        "poem": response.content
    }

# Node 2: Generate Joke from Poem
def generate_joke(state: State):
    response = llm.invoke(
        f"Create a funny joke inspired by this poem:\n\n{state['poem']}"
    )

    return {
        "joke": response.content
    }

# Build Graph
builder = StateGraph(State)

builder.add_node("generate_poem", generate_poem)
builder.add_node("generate_joke", generate_joke)

builder.add_edge(START, "generate_poem")
builder.add_edge("generate_poem", "generate_joke")
builder.add_edge("generate_joke", END)

# Memory
memory = InMemorySaver()

graph = builder.compile(checkpointer=memory)

# Run
config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode=["messages", "updates", "values", "checkpoints"],
    version="v2"
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        node_name = metadata.get("langgraph_node")
        
        if msg.content:
            if node_name == "generate_poem":
                print(f"\n[Poem Node] {msg.content}", end="", flush=True)
            elif node_name == "generate_joke":
                print(f"\n[Joke Node] {msg.content}", end="", flush=True)

    elif chunk["type"] == "updates":
        for node_name, updates in chunk["data"].items():
            print(f"\n[UPDATES] Node `{node_name}` completed with updates: {list(updates.keys())}"
)
    elif chunk["type"] == "values":
        print(f"\n[VALUES] Current State Snapshot: {chunk['data']}")
    elif chunk["type"] == "checkpoints":
        snapshot = chunk["data"]
        print(f"\n[CHECKPOINT] Saved with ID: {snapshot['config']['configurable'].get('checkpoint_id')}, Next Node: {snapshot.get('next', ())}")
        

[CHECKPOINT] Saved with ID: 1f15f3c5-2092-67c7-bfff-0ab39731fc37, Next Node: ['__start__']

[VALUES] Current State Snapshot: {'topic': 'ice cream'}

[CHECKPOINT] Saved with ID: 1f15f3c5-2099-6d14-8000-b20dc3222060, Next Node: ['generate_poem']

[Poem Node] **Scoops of Summer

[Poem Node] **

A swirl of pink, a scoop of gold,
A treat that's better cold than cold.
It drips down fingers

...

[Joke Node] 's defense? *"Look, I gave you a swirl of gold and a whole world of

[Joke Node]  flavor. What more do you want — a waffle cone of emotional commitment?"*

🍦😄

[UPDATES] Node `generate_joke` completed with updates: ['joke']

[VALUES] Current State Snapshot: {'topic': 'ice cream', 'poem': "**Scoops of Summer**\n\nA swirl of pink, a scoop 
of gold,\nA treat that's better cold than cold.\nIt drips down fingers, sweet and slow,\nA fleeting joy we all must
know.\n\nFrom waffle cones to paper cups,\nIt lifts our spirits, fills us up.\nIn every flavor, every hue,\nA small
delight — the whole world through.", 'joke': '**Here\'s a joke for you:**\n\nWhy did the ice cream file a complaint
against the summer sun?\n\nBecause it promised a *long-term relationship*… but the sun kept getting too hot and 
cold, the ice cream kept *dripping* with emotional baggage, and by the end, it was just a sticky mess on the 
sidewalk saying, *"I thought we had something — but turns out, I was just a fleeting joy in your orbit."*\n\nThe 
sun\'s defense? *"Look, I gave you a swirl of gold and a whole world of flavor. What more do you want — a waffle 
cone of emotional commitment?"*\n\n🍦😄'}

[CHECKPOINT] Saved with ID: 1f15f3c6-33a2-6eb7-8002-0f18e94881aa, Next Node: []